# 01 — 数据理解与 Order-Level Master Table

## 本 notebook 要完成什么

这一本 notebook 是整个 **Olist post-purchase CX redesign** 项目的数据基础层，不负责证明“晚到是否伤害 CX”，也不负责提出服务方案。

本 notebook 的目标是：

1. 读取并审计 9 张 Olist raw CSV；
2. 明确每张表的 grain、key、missingness 与 join relationship；
3. 找出会造成订单重复的 one-to-many 关系；
4. 在 merge 前先对 item / payment / review 数据聚合或去重；
5. 构建 **one row per `order_id`** 的 `master_order_table`；
6. 单独建立后续 late-delivery 分析可用的 `analysis_orders` population；
7. 用 sanity checks 验证 row count、uniqueness、join fan-out 与关键字段；
8. 把审计表和 processed data 保存到项目规定的文件夹。

### Evidence boundary

这一本 notebook 属于 **Level 1: Direct data analysis / data processing**。

它可以支持关于数据结构、数据质量、join 规则和分析 population 的陈述；它**不能**证明 late delivery 导致低评分，也不能证明任何未来 service recovery intervention 会有效。

## 与 Gap Matrix 和参考项目的关系

本 notebook 主要借鉴参考项目中的**方法逻辑**，而不是复制代码：

- **Zeash**：grain-aware audit、aggregate-before-join、review deterministic dedupe、calendar-date late definition、assertion checks；
- **Moses**：简单清晰的 schema / missing-value inspection，以及先报告缺失再过滤的顺序。

本项目在这些基础上增加：

- 一个服务于整个 CX redesign pipeline 的项目级 `master_order_table`；
- 明确的 **join contract** 和 evidence boundary；
- 为后续 seller/category 分析保留 `order_seller_bridge` 与 `order_category_bridge`，避免把 multi-seller / multi-category 订单错误归因给单一标签；
- 明确指出未来 prediction 场景中的 post-outcome data leakage 风险。

核心原则：

> **Never join a raw one-to-many table directly to the order-level base table.**

## 0. Setup

代码使用相对项目路径，而不是写死本机绝对路径。Notebook 无论从项目根目录还是 `notebooks/` 目录运行，都应能自动找到 `data/raw/`。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def find_project_root(start=None):
    """Find the nearest parent directory that contains data/raw."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Expected a data/raw folder in the current directory or a parent directory."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for folder in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root detected:", PROJECT_ROOT.name)
print("Raw data folder:", RAW_DIR.relative_to(PROJECT_ROOT))
print("pandas version:", pd.__version__)

Project root detected: olist-post-purchase-cx-redesign
Raw data folder: data/raw
pandas version: 3.0.5


## 1. Load the 9 raw tables

这里先把 9 张原始表读入，并在读取阶段把关键 timestamp columns 转成 datetime。

此时**不做 merge**。原因是不同表的 grain 不同，必须先检查 cardinality。

In [2]:
FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

missing_files = [filename for filename in FILES.values() if not (RAW_DIR / filename).exists()]
assert not missing_files, f"Missing raw files: {missing_files}"

customers = pd.read_csv(RAW_DIR / FILES["customers"])
geolocation = pd.read_csv(RAW_DIR / FILES["geolocation"])
order_items = pd.read_csv(
    RAW_DIR / FILES["order_items"],
    parse_dates=["shipping_limit_date"],
)
order_payments = pd.read_csv(RAW_DIR / FILES["order_payments"])
order_reviews = pd.read_csv(
    RAW_DIR / FILES["order_reviews"],
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)
orders = pd.read_csv(
    RAW_DIR / FILES["orders"],
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)
products = pd.read_csv(RAW_DIR / FILES["products"])
sellers = pd.read_csv(RAW_DIR / FILES["sellers"])
category_translation = pd.read_csv(RAW_DIR / FILES["category_translation"])

tables = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}

print("Loaded tables:")
for name, df in tables.items():
    print(f"- {name:22s} {df.shape}")

Loaded tables:
- customers              (99441, 5)
- geolocation            (1000163, 5)
- order_items            (112650, 7)
- order_payments         (103886, 5)
- order_reviews          (99224, 7)
- orders                 (99441, 8)
- products               (32951, 9)
- sellers                (3095, 4)
- category_translation   (71, 2)


## 2. Table inventory 与 missing-value audit

这一节要回答两个基础问题：

1. 每张表有多少 rows / columns，关键 key 是否唯一？
2. 哪些字段有 missing values，missing 的规模是多少？

注意：missing 不会在这里被自动填补。对于 post-purchase 分析，缺失本身可能决定某条记录是否有资格进入后续分析 population，因此先报告、后决定。

In [3]:
EXPECTED_GRAIN = {
    "orders": "one row per order",
    "customers": "one row per customer_id (customer_id is order-linked, not person-level)",
    "order_items": "one row per order item",
    "order_payments": "one row per payment record / sequence",
    "order_reviews": "one row per review record",
    "products": "one row per product",
    "sellers": "one row per seller",
    "category_translation": "one row per category mapping",
    "geolocation": "multiple observations per zip-code prefix",
}

KEY_COLUMN = {
    "orders": "order_id",
    "customers": "customer_id",
    "order_items": "order_id",
    "order_payments": "order_id",
    "order_reviews": "order_id",
    "products": "product_id",
    "sellers": "seller_id",
    "category_translation": "product_category_name",
    "geolocation": "geolocation_zip_code_prefix",
}

inventory_rows = []
missing_rows = []

for name, df in tables.items():
    key = KEY_COLUMN[name]
    inventory_rows.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "expected_grain": EXPECTED_GRAIN[name],
        "key_checked": key,
        "distinct_key_values": df[key].nunique(dropna=False),
        "extra_rows_beyond_distinct_key": len(df) - df[key].nunique(dropna=False),
        "full_row_duplicates": int(df.duplicated().sum()),
        "column_names": ", ".join(df.columns),
    })

    for column in df.columns:
        missing_count = int(df[column].isna().sum())
        missing_rows.append({
            "table": name,
            "column": column,
            "dtype": str(df[column].dtype),
            "missing_count": missing_count,
            "missing_pct": missing_count / len(df) if len(df) else np.nan,
        })

table_inventory = pd.DataFrame(inventory_rows).sort_values("table").reset_index(drop=True)
missing_value_summary = pd.DataFrame(missing_rows).sort_values(
    ["missing_count", "table", "column"], ascending=[False, True, True]
).reset_index(drop=True)

table_inventory.to_csv(TABLE_DIR / "table_inventory.csv", index=False)
missing_value_summary.to_csv(TABLE_DIR / "missing_value_summary.csv", index=False)

print("Table inventory")
display(table_inventory)

print("Columns with missing values")
display(missing_value_summary.query("missing_count > 0"))

Table inventory


,table,rows,columns,expected_grain,key_checked,distinct_key_values,extra_rows_beyond_distinct_key,full_row_duplicates,column_names
0,category_translation,71,2,one row per category mapping,product_category_name,71,0,0,"product_category_name, product_category_name_english"
1,customers,99441,5,"one row per customer_id (customer_id is order-linked, not person-level)",customer_id,99441,0,0,"customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state"
2,geolocation,1000163,5,multiple observations per zip-code prefix,geolocation_zip_code_prefix,19015,981148,261831,"geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state"
3,order_items,112650,7,one row per order item,order_id,98666,13984,0,"order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value"
4,order_payments,103886,5,one row per payment record / sequence,order_id,99440,4446,0,"order_id, payment_sequential, payment_type, payment_installments, payment_value"
5,order_reviews,99224,7,one row per review record,order_id,98673,551,0,"review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer..."
6,orders,99441,8,one row per order,order_id,99441,0,0,"order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, orde..."
7,products,32951,9,one row per product,product_id,32951,0,0,"product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weig..."
8,sellers,3095,4,one row per seller,seller_id,3095,0,0,"seller_id, seller_zip_code_prefix, seller_city, seller_state"


Columns with missing values


,table,column,dtype,missing_count,missing_pct
0,order_reviews,review_comment_title,str,87656,0.883
1,order_reviews,review_comment_message,str,58247,0.587
2,orders,order_delivered_customer_date,datetime64[us],2965,0.030
3,orders,order_delivered_carrier_date,datetime64[us],1783,0.018
4,products,product_category_name,str,610,0.019
5,products,product_description_lenght,float64,610,0.019
6,products,product_name_lenght,float64,610,0.019
7,products,product_photos_qty,float64,610,0.019
8,orders,order_approved_at,datetime64[us],160,0.002
9,products,product_height_cm,float64,2,0.000


### 这一节应该如何解读

`orders` 的 `order_id` 是否唯一，是整本 notebook 最关键的 base-grain check。与此同时，`order_items`、`order_payments`、`order_reviews` 的 `order_id` 出现重复并不是“脏数据”本身，而是它们天然处在更细的 grain。

因此，看到这些重复后不应该直接 `drop_duplicates(order_id)`；正确做法是先理解每张表一行代表什么，再决定 aggregate 或 deterministic dedupe 规则。

## 3. Grain、key relationship 与 join fan-out audit

这一节明确哪些关系可以安全 join，哪些关系会造成 fan-out。

重点检查：

- `orders` 是否 one row per `order_id`；
- `orders` 与 `customers` 是否在 `customer_id` 上 1:1；
- `customer_unique_id` 是否确实比 `customer_id` 更接近 person-level identifier；
- 一单可以有多少 item / seller / payment / review；
- duplicate reviews 是否存在 score conflict；
- geolocation 是否适合直接按 zip prefix join。

In [4]:
# Base-grain checks
assert orders["order_id"].is_unique, "orders.order_id must be unique"
assert customers["customer_id"].is_unique, "customers.customer_id must be unique"
assert products["product_id"].is_unique, "products.product_id must be unique"
assert sellers["seller_id"].is_unique, "sellers.seller_id must be unique"
assert category_translation["product_category_name"].is_unique, "category mapping key must be unique"

orders_without_customer = (~orders["customer_id"].isin(customers["customer_id"])).sum()
customers_without_order = (~customers["customer_id"].isin(orders["customer_id"])).sum()
assert orders_without_customer == 0
assert customers_without_order == 0

item_per_order = (
    order_items.groupby("order_id")
    .agg(
        item_rows=("order_item_id", "size"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
    )
)

payment_per_order = (
    order_payments.groupby("order_id")
    .agg(
        payment_rows=("payment_sequential", "size"),
        payment_types=("payment_type", "nunique"),
    )
)

review_per_order = (
    order_reviews.groupby("order_id")
    .agg(
        review_rows=("review_id", "size"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max"),
    )
)

review_id_stats = (
    order_reviews.groupby("review_id")
    .agg(review_rows=("order_id", "size"), unique_orders=("order_id", "nunique"))
)

geolocation_rows_per_zip = geolocation.groupby("geolocation_zip_code_prefix").size()

join_cardinality_rows = [
    {"check": "orders rows", "value": len(orders), "interpretation": "Base order population"},
    {"check": "unique order_id in orders", "value": orders["order_id"].nunique(), "interpretation": "Should equal orders rows"},
    {"check": "duplicate order_id rows in orders", "value": len(orders) - orders["order_id"].nunique(), "interpretation": "Must be 0"},
    {"check": "unique customer_id", "value": customers["customer_id"].nunique(), "interpretation": "Order-linked customer key"},
    {"check": "unique customer_unique_id", "value": customers["customer_unique_id"].nunique(), "interpretation": "Person-level key for future repeat-purchase analysis"},
    {"check": "orders without matching customer", "value": int(orders_without_customer), "interpretation": "Must be 0 for a complete customer join"},
    {"check": "orders with >1 item row", "value": int((item_per_order["item_rows"] > 1).sum()), "interpretation": "Raw item join would fan out these orders"},
    {"check": "max item rows on one order", "value": int(item_per_order["item_rows"].max()), "interpretation": "Upper bound of item-level fan-out"},
    {"check": "orders with >1 seller", "value": int((item_per_order["unique_sellers"] > 1).sum()), "interpretation": "Single-seller attribution is unsafe for these orders"},
    {"check": "max sellers on one order", "value": int(item_per_order["unique_sellers"].max()), "interpretation": "Supports later order-seller bridge"},
    {"check": "orders with >1 payment row", "value": int((payment_per_order["payment_rows"] > 1).sum()), "interpretation": "Raw payment join would fan out these orders"},
    {"check": "max payment rows on one order", "value": int(payment_per_order["payment_rows"].max()), "interpretation": "Upper bound of payment fan-out"},
    {"check": "orders with >1 review row", "value": int((review_per_order["review_rows"] > 1).sum()), "interpretation": "Reviews require a deterministic order-level rule"},
    {"check": "multi-review orders with conflicting scores", "value": int(((review_per_order["review_rows"] > 1) & (review_per_order["min_review_score"] != review_per_order["max_review_score"])).sum()), "interpretation": "Arbitrary first-row dedupe could change the score"},
    {"check": "review_id values spanning >1 order", "value": int(((review_id_stats["review_rows"] > 1) & (review_id_stats["unique_orders"] > 1)).sum()), "interpretation": "Some review IDs are shared across orders; later review analysis must note non-independence"},
    {"check": "geolocation zip prefixes with >1 row", "value": int((geolocation_rows_per_zip > 1).sum()), "interpretation": "Raw geolocation is many-to-one by zip prefix and should not be directly joined to order grain"},
]

join_cardinality_audit = pd.DataFrame(join_cardinality_rows)
display(join_cardinality_audit)

,check,value,interpretation
0,orders rows,99441,Base order population
1,unique order_id in orders,99441,Should equal orders rows
2,duplicate order_id rows in orders,0,Must be 0
3,unique customer_id,99441,Order-linked customer key
4,unique customer_unique_id,96096,Person-level key for future repeat-purchase analysis
5,orders without matching customer,0,Must be 0 for a complete customer join
6,orders with >1 item row,9803,Raw item join would fan out these orders
7,max item rows on one order,21,Upper bound of item-level fan-out
8,orders with >1 seller,1278,Single-seller attribution is unsafe for these orders
9,max sellers on one order,5,Supports later order-seller bridge


### Grain audit 的项目含义

这一节不是在找业务 insight，而是在建立后续分析可以依赖的 **data contract**：

- `orders` 是 order-level base；
- `order_items` 和 `order_payments` 必须先 aggregate；
- `order_reviews` 必须先按明确规则 dedupe；
- seller/category 不是天然 one-to-one，因此未来 seller/category risk 不应该只依赖一个“primary seller/category”；
- raw geolocation 由于一个 zip prefix 对应多行，V1 不把它直接 merge 到 master。

这正是本 notebook 对整个项目最重要的贡献：**先保护分析 grain，再讨论业务结果。**

## 4. Enrich items，并建立 order-level aggregates 与 bridge tables

`order_items` 是后续 seller/category 分析的重要来源，但它本身是 item-level。

V1 采取两层输出：

1. **order-level aggregation**：进入 `master_order_table`，只保留 count / value 等不会造成错误归因的字段；
2. **bridge tables**：保留真实的 order–seller 与 order–category 多对多关系，供后续风险分析使用。

这样不需要人为选一个“primary seller”来代表 multi-seller order。

In [5]:
# Product category enrichment: many products -> one translation row per category
products_enriched = products.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="m:1",
)

# If an English mapping is missing, retain the original Portuguese category;
# if the product category itself is missing, label it explicitly as unknown.
products_enriched["product_category_clean"] = (
    products_enriched["product_category_name_english"]
    .fillna(products_enriched["product_category_name"])
    .fillna("unknown")
)

items_enriched = order_items.merge(
    products_enriched[["product_id", "product_category_clean"]],
    on="product_id",
    how="left",
    validate="m:1",
).merge(
    sellers[["seller_id", "seller_city", "seller_state"]],
    on="seller_id",
    how="left",
    validate="m:1",
)

assert len(items_enriched) == len(order_items), "Item enrichment changed the number of item rows"
assert items_enriched["product_category_clean"].notna().all(), "Every item should have a clean category label"
assert items_enriched["seller_id"].notna().all(), "Every item should retain a seller_id"

order_item_agg = (
    items_enriched.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "size"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_clean", "nunique"),
        total_item_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
    )
)
order_item_agg["is_multi_seller"] = order_item_agg["seller_count"] > 1
order_item_agg["is_multi_category"] = order_item_agg["category_count"] > 1

assert order_item_agg["order_id"].is_unique

# Preserve order–seller grain for later seller-risk work.
order_seller_bridge = (
    items_enriched.groupby(
        ["order_id", "seller_id", "seller_city", "seller_state"],
        as_index=False,
        dropna=False,
    )
    .agg(
        seller_item_count=("order_item_id", "size"),
        seller_item_value=("price", "sum"),
        seller_freight_value=("freight_value", "sum"),
    )
)

# Preserve order–category grain for later category-risk work.
order_category_bridge = (
    items_enriched.groupby(["order_id", "product_category_clean"], as_index=False, dropna=False)
    .agg(
        category_item_count=("order_item_id", "size"),
        category_item_value=("price", "sum"),
        category_freight_value=("freight_value", "sum"),
    )
)

# Add category multiplicity to the audit table now that product enrichment is available.
join_cardinality_audit = pd.concat(
    [
        join_cardinality_audit,
        pd.DataFrame([
            {
                "check": "orders with >1 category",
                "value": int((order_item_agg["category_count"] > 1).sum()),
                "interpretation": "Single-category attribution is unsafe for these orders",
            },
            {
                "check": "products with missing raw category",
                "value": int(products["product_category_name"].isna().sum()),
                "interpretation": "Retained as explicit 'unknown' category",
            },
            {
                "check": "products with raw category but no English translation",
                "value": int((products_enriched["product_category_name"].notna() & products_enriched["product_category_name_english"].isna()).sum()),
                "interpretation": "Falls back to the original Portuguese category name",
            },
        ])
    ],
    ignore_index=True,
)

join_cardinality_audit.to_csv(TABLE_DIR / "join_cardinality_audit.csv", index=False)

display(order_item_agg.head())
print("order_seller_bridge rows:", len(order_seller_bridge))
print("order_category_bridge rows:", len(order_category_bridge))

,order_id,item_count,product_count,seller_count,category_count,total_item_price,total_freight_value,is_multi_seller,is_multi_category
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,1,58.900,13.290,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,1,239.900,19.930,False,False
2,000229ec398224ef6ca0657da4fc703e,1,1,1,1,199.000,17.870,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,1,12.990,12.790,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,1,199.900,18.140,False,False


order_seller_bridge rows: 100010
order_category_bridge rows: 99470


## 5. Aggregate payments 与 deterministic review dedupe

### Payments

一单可能有多条 payment records，因此这里先聚合成 one row per order，再进入 master。

### Reviews

一单也可能有多条 review records。V1 使用一个可复现规则：

> 对每个 `order_id`，保留 `review_answer_timestamp` 最新的记录；如果时间完全相同，则用 `review_id` 升序作为 tie-breaker。

这个规则的目的只是建立 reproducible one-review-per-order representation。它**不代表**“最新 review 一定是客户最真实或最终的意见”。

In [6]:
# Payment aggregation
payment_agg = (
    order_payments.groupby("order_id", as_index=False)
    .agg(
        payment_record_count=("payment_sequential", "size"),
        payment_type_count=("payment_type", "nunique"),
        total_payment_value=("payment_value", "sum"),
        max_payment_installments=("payment_installments", "max"),
    )
)

payment_types = (
    order_payments.groupby("order_id")["payment_type"]
    .agg(lambda s: " | ".join(sorted(set(s))))
    .rename("payment_types")
    .reset_index()
)
payment_agg = payment_agg.merge(payment_types, on="order_id", how="left", validate="1:1")
assert payment_agg["order_id"].is_unique

# Review multiplicity audit before dedupe
review_counts = order_reviews.groupby("order_id").size().rename("review_record_count")

# Stable sort makes the rule deterministic even if future data contains ties.
reviews_deduped = (
    order_reviews.sort_values(
        ["order_id", "review_answer_timestamp", "review_id"],
        ascending=[True, False, True],
        kind="mergesort",
    )
    .drop_duplicates("order_id", keep="first")
    .merge(review_counts.reset_index(), on="order_id", how="left", validate="1:1")
)
reviews_deduped["has_multiple_reviews"] = reviews_deduped["review_record_count"] > 1

assert reviews_deduped["order_id"].is_unique
assert len(reviews_deduped) == order_reviews["order_id"].nunique()
assert reviews_deduped["review_score"].between(1, 5).all()

# Check whether the review_id tie-breaker was needed at the maximum timestamp.
max_answer_time = order_reviews.groupby("order_id")["review_answer_timestamp"].transform("max")
latest_rows = order_reviews[order_reviews["review_answer_timestamp"].eq(max_answer_time)]
orders_needing_review_tiebreak = int((latest_rows.groupby("order_id").size() > 1).sum())

print("Payment rows before aggregation:", len(order_payments))
print("Payment order rows after aggregation:", len(payment_agg))
print("Review rows before dedupe:", len(order_reviews))
print("Review order rows after dedupe:", len(reviews_deduped))
print("Orders where review_id tie-breaker was needed:", orders_needing_review_tiebreak)

Payment rows before aggregation: 103886
Payment order rows after aggregation: 99440
Review rows before dedupe: 99224
Review order rows after dedupe: 98673
Orders where review_id tie-breaker was needed: 0


### Dedupe / aggregation 风险说明

这里有两个需要长期保留的 limitation：

1. **review dedupe 是分析规则，不是事实真相**。后续 review impact analysis 应该说明该规则；
2. 某些 `review_id` 会跨多个 `order_id` 出现，因此这些 review observations 并不完全独立。Notebook 03 做统计解释时应把它作为限制或 sensitivity check。

## 6. Build the one-row-per-order master table

Master table 的 base 是 `orders`。其他表只能以已经处理成 unique key 的形态 left join 进来。

采用 left join 的原因是：master 应尽量保留完整 order population，而不是因为没有 item/payment/review record 就提前删掉订单。

In [7]:
def checked_left_merge(left, right, on, validate, label):
    """Left join while protecting the order-level row count and order_id uniqueness."""
    before_rows = len(left)
    merged = left.merge(right, on=on, how="left", validate=validate)
    assert len(merged) == before_rows, f"{label}: row count changed from {before_rows} to {len(merged)}"
    assert merged["order_id"].is_unique, f"{label}: order_id is no longer unique"
    print(f"PASS - {label}: {len(merged):,} rows, one row per order_id")
    return merged


master_order_table = orders.copy()

master_order_table = checked_left_merge(
    master_order_table,
    customers,
    on="customer_id",
    validate="1:1",
    label="orders + customers",
)

master_order_table = checked_left_merge(
    master_order_table,
    order_item_agg,
    on="order_id",
    validate="1:1",
    label="+ order-level item aggregates",
)

master_order_table = checked_left_merge(
    master_order_table,
    payment_agg,
    on="order_id",
    validate="1:1",
    label="+ order-level payment aggregates",
)

master_order_table = checked_left_merge(
    master_order_table,
    reviews_deduped,
    on="order_id",
    validate="1:1",
    label="+ deduplicated review",
)

# Explicit coverage flags are safer than silently replacing missing aggregates with zero.
master_order_table["has_item_record"] = master_order_table["item_count"].notna()
master_order_table["has_payment_record"] = master_order_table["payment_record_count"].notna()
master_order_table["has_review_record"] = master_order_table["review_score"].notna()

# Keep count fields as nullable integers so missing source records remain visible.
nullable_int_columns = [
    "item_count",
    "product_count",
    "seller_count",
    "category_count",
    "payment_record_count",
    "payment_type_count",
    "max_payment_installments",
    "review_record_count",
]
for column in nullable_int_columns:
    master_order_table[column] = master_order_table[column].astype("Int64")

print("Master shape:", master_order_table.shape)
print("Unique order_id:", master_order_table["order_id"].nunique())
display(master_order_table.head())

PASS - orders + customers: 99,441 rows, one row per order_id
PASS - + order-level item aggregates: 99,441 rows, one row per order_id
PASS - + order-level payment aggregates: 99,441 rows, one row per order_id
PASS - + deduplicated review: 99,441 rows, one row per order_id
Master shape: (99441, 36)
Unique order_id: 99441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,item_count,product_count,seller_count,category_count,total_item_price,total_freight_value,is_multi_seller,is_multi_category,payment_record_count,payment_type_count,total_payment_value,max_payment_installments,payment_types,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_record_count,has_multiple_reviews,has_item_record,has_payment_record,has_review_record
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,1,1,1,29.990,8.720,False,False,3,2,38.710,1,credit_card | voucher,a54f0611adc9ed256b57ede6b6eb5114,4.000,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danific...",2017-10-11,2017-10-12 03:43:48,1,False,True,True,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,1,1,1,118.700,22.760,False,False,1,1,141.460,1,boleto,8d5266042046a06655c8db133d120ba5,4.000,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,1,False,True,True,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,1,1,1,159.900,19.220,False,False,1,1,179.120,3,credit_card,e73b67b67587f7644d5bd1a52deb1b01,5.000,NaN,NaN,2018-08-18,2018-08-22 19:07:58,1,False,True,True,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1,1,1,1,45.000,27.200,False,False,1,1,72.200,1,credit_card,359d03e676b3c069f62cadba8dd3f6e8,5.000,NaN,O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista.,2017-12-03,2017-12-05 19:21:58,1,False,True,True,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1,1,1,1,19.900,8.720,False,False,1,1,28.620,1,credit_card,e50934924e227544ba8246aeb3770dd4,5.000,NaN,NaN,2018-02-17,2018-02-18 13:02:51,1,False,True,True,True


### Merge 之后的 sanity check 逻辑

每一次 join 后都立即检查：

- row count 没有增加；
- `order_id` 仍然唯一。

这比在最后才发现重复更安全，因为一旦某个 merge 发生 fan-out，可以立刻定位是哪一步破坏了 grain。

同时这里**不把缺失的 item/payment/review aggregate 自动填成 0**，而是增加 `has_*_record` flags。这样“没有来源记录”和“真实数值为 0”不会被混在一起。

## 7. Define the delivery-analysis population

后续 late-delivery 分析只对能够可靠比较 actual vs estimated delivery date 的订单成立。

V1 eligibility：

```text
order_status == "delivered"
AND order_delivered_customer_date is not null
AND order_estimated_delivery_date is not null
```

Late definition 使用 **calendar-date comparison**：

```text
actual delivery calendar date > estimated delivery calendar date
```

这样可以避免 `order_estimated_delivery_date` 存在午夜时间戳时，把“承诺日当天晚些时候送达”的订单误判为 late。

本 notebook 只负责定义和验证字段；late rate、severity distribution 和 CX impact 留到 Notebook 02/03。

In [8]:
# Keep normalized calendar dates as explicit fields.
master_order_table["actual_delivery_date"] = master_order_table["order_delivered_customer_date"].dt.normalize()
master_order_table["estimated_delivery_date"] = master_order_table["order_estimated_delivery_date"].dt.normalize()

eligible = (
    master_order_table["order_status"].eq("delivered")
    & master_order_table["actual_delivery_date"].notna()
    & master_order_table["estimated_delivery_date"].notna()
)
master_order_table["delivery_analysis_eligible"] = eligible

calendar_delay_days = (
    master_order_table["actual_delivery_date"] - master_order_table["estimated_delivery_date"]
).dt.days.astype("Int64")
master_order_table["delay_days"] = calendar_delay_days.where(eligible)

calendar_late = master_order_table["actual_delivery_date"] > master_order_table["estimated_delivery_date"]
master_order_table["is_late"] = calendar_late.where(eligible).astype("boolean")

# Retain the naive timestamp rule only as a sensitivity field; it is not the primary definition.
timestamp_late = (
    master_order_table["order_delivered_customer_date"]
    > master_order_table["order_estimated_delivery_date"]
)
master_order_table["is_late_timestamp_rule"] = timestamp_late.where(eligible).astype("boolean")

# Severity bands are created here for consistency, but interpreted in Notebook 02.
delay_band = pd.Series(pd.NA, index=master_order_table.index, dtype="string")
delay_band.loc[eligible & (master_order_table["delay_days"] <= 0)] = "on time or early"
delay_band.loc[eligible & master_order_table["delay_days"].between(1, 3)] = "1-3 days late"
delay_band.loc[eligible & master_order_table["delay_days"].between(4, 7)] = "4-7 days late"
delay_band.loc[eligible & master_order_table["delay_days"].between(8, 14)] = "8-14 days late"
delay_band.loc[eligible & master_order_table["delay_days"].between(15, 30)] = "15-30 days late"
delay_band.loc[eligible & (master_order_table["delay_days"] > 30)] = "31+ days late"
master_order_table["delay_band"] = delay_band

analysis_orders = master_order_table.loc[eligible].copy()

# Status summary makes the population boundary visible rather than silently filtering rows.
order_status_summary = (
    orders.groupby("order_status", as_index=False)
    .agg(
        orders=("order_id", "size"),
        with_delivery_date=("order_delivered_customer_date", "count"),
    )
)
order_status_summary["without_delivery_date"] = (
    order_status_summary["orders"] - order_status_summary["with_delivery_date"]
)
order_status_summary["pct_orders"] = order_status_summary["orders"] / len(orders)
order_status_summary = order_status_summary.sort_values("orders", ascending=False).reset_index(drop=True)
order_status_summary.to_csv(TABLE_DIR / "order_status_summary.csv", index=False)

same_day_timestamp_false_late = int(
    ((analysis_orders["is_late_timestamp_rule"] == True) & (analysis_orders["is_late"] == False)).sum()
)

print("Delivery-analysis eligible orders:", f"{len(analysis_orders):,}")
print("Same-day orders that naive timestamp rule would label late:", f"{same_day_timestamp_false_late:,}")
display(order_status_summary)

Delivery-analysis eligible orders: 96,470
Same-day orders that naive timestamp rule would label late: 1,292


,order_status,orders,with_delivery_date,without_delivery_date,pct_orders
0,delivered,96478,96470,8,0.970
1,shipped,1107,0,1107,0.011
2,canceled,625,6,619,0.006
3,unavailable,609,0,609,0.006
4,invoiced,314,0,314,0.003
5,processing,301,0,301,0.003
6,created,5,0,5,0.000
7,approved,2,0,2,0.000


### Population boundary 的解释

`analysis_orders` 是为了回答“在有可观察 delivery outcome 的 delivered orders 中，哪些订单 late”这一类问题。

它**不是**“所有 CX experiences”的替代品。未送达、取消、缺少 actual delivery timestamp 的订单可能同样代表差体验，但无法用同一个 `delay_days` 定义衡量，因此这里不做插补，也不把它们假装成 on-time。

## 8. Data leakage、duplicate row 与 attribution 风险

在后续 notebook 中需要持续遵守以下边界：

- **Post-outcome leakage**：`review_score`、review text、`order_delivered_customer_date`、`delay_days`、`is_late` 都是在结果发生后才能知道的信息。如果未来设计 pre-delivery delay-risk model，这些字段不能作为预测输入；
- **Multi-seller attribution**：一个订单可以包含多个 seller。seller risk 应使用 `order_seller_bridge`，不能把某个随机/primary seller 当成整个订单的唯一责任方；
- **Multi-category attribution**：同理，category analysis 应使用 `order_category_bridge` 或明确的 order-category grain；
- **Review dedupe**：latest-answer rule 是 reproducibility choice，不是客户心理状态的事实；
- **Missingness**：Notebook 01 报告 missing，并只在分析定义明确要求时过滤，不假设 missing at random；
- **Causality**：这里构建的数据层只能支持后续 observational association analysis，不能自动转化为因果结论。

## 9. Final validation and save outputs

最后用一张 validation table 把 notebook 的 acceptance criteria 显式化。

只要任何关键 check 失败，代码会 `assert` 并停止，而不是继续生成看似正常但 grain 已经错误的结果。

In [9]:
validation_rows = []


def add_validation(check, value, expected, passed):
    validation_rows.append({
        "check": check,
        "value": value,
        "expected": expected,
        "status": "PASS" if bool(passed) else "FAIL",
    })


add_validation("source orders rows", len(orders), "99441 for this snapshot / equals unique order_id", len(orders) == orders["order_id"].nunique())
add_validation("master rows equal source orders", len(master_order_table), len(orders), len(master_order_table) == len(orders))
add_validation("master unique order_id", master_order_table["order_id"].nunique(), len(master_order_table), master_order_table["order_id"].is_unique)
add_validation("master duplicate order_id rows", int(master_order_table["order_id"].duplicated().sum()), 0, master_order_table["order_id"].duplicated().sum() == 0)
add_validation("customer join missing customer_unique_id", int(master_order_table["customer_unique_id"].isna().sum()), 0, master_order_table["customer_unique_id"].isna().sum() == 0)
add_validation("order_item_agg unique order_id", int(order_item_agg["order_id"].duplicated().sum()), 0, order_item_agg["order_id"].is_unique)
add_validation("payment_agg unique order_id", int(payment_agg["order_id"].duplicated().sum()), 0, payment_agg["order_id"].is_unique)
add_validation("reviews_deduped unique order_id", int(reviews_deduped["order_id"].duplicated().sum()), 0, reviews_deduped["order_id"].is_unique)
add_validation("review scores outside 1-5", int((master_order_table["review_score"].dropna().between(1, 5) == False).sum()), 0, master_order_table["review_score"].dropna().between(1, 5).all())
add_validation("analysis rows equal eligible flag", len(analysis_orders), int(master_order_table["delivery_analysis_eligible"].sum()), len(analysis_orders) == int(master_order_table["delivery_analysis_eligible"].sum()))
add_validation("analysis unique order_id", analysis_orders["order_id"].nunique(), len(analysis_orders), analysis_orders["order_id"].is_unique)
add_validation("analysis non-delivered rows", int((analysis_orders["order_status"] != "delivered").sum()), 0, (analysis_orders["order_status"] == "delivered").all())
add_validation("analysis missing actual delivery date", int(analysis_orders["actual_delivery_date"].isna().sum()), 0, analysis_orders["actual_delivery_date"].notna().all())
add_validation("analysis missing estimated delivery date", int(analysis_orders["estimated_delivery_date"].isna().sum()), 0, analysis_orders["estimated_delivery_date"].notna().all())
add_validation("analysis missing delay_days", int(analysis_orders["delay_days"].isna().sum()), 0, analysis_orders["delay_days"].notna().all())

master_table_validation = pd.DataFrame(validation_rows)
display(master_table_validation)

assert master_table_validation["status"].eq("PASS").all(), "One or more master-table validation checks failed"

# Save processed datasets only after all validations pass.
master_order_table.to_csv(PROCESSED_DIR / "master_order_table.csv", index=False)
analysis_orders.to_csv(PROCESSED_DIR / "analysis_orders.csv", index=False)
order_seller_bridge.to_csv(PROCESSED_DIR / "order_seller_bridge.csv", index=False)
order_category_bridge.to_csv(PROCESSED_DIR / "order_category_bridge.csv", index=False)
master_table_validation.to_csv(TABLE_DIR / "master_table_validation.csv", index=False)

saved_outputs = [
    TABLE_DIR / "table_inventory.csv",
    TABLE_DIR / "missing_value_summary.csv",
    TABLE_DIR / "join_cardinality_audit.csv",
    TABLE_DIR / "order_status_summary.csv",
    TABLE_DIR / "master_table_validation.csv",
    PROCESSED_DIR / "master_order_table.csv",
    PROCESSED_DIR / "analysis_orders.csv",
    PROCESSED_DIR / "order_seller_bridge.csv",
    PROCESSED_DIR / "order_category_bridge.csv",
]

print("All validation checks passed. Saved outputs:")
for path in saved_outputs:
    assert path.exists(), f"Expected output was not created: {path}"
    print("-", path.relative_to(PROJECT_ROOT))

,check,value,expected,status
0,source orders rows,99441,99441 for this snapshot / equals unique order_id,PASS
1,master rows equal source orders,99441,99441,PASS
2,master unique order_id,99441,99441,PASS
3,master duplicate order_id rows,0,0,PASS
4,customer join missing customer_unique_id,0,0,PASS
5,order_item_agg unique order_id,0,0,PASS
6,payment_agg unique order_id,0,0,PASS
7,reviews_deduped unique order_id,0,0,PASS
8,review scores outside 1-5,0,0,PASS
9,analysis rows equal eligible flag,96470,96470,PASS


All validation checks passed. Saved outputs:
- outputs/tables/table_inventory.csv
- outputs/tables/missing_value_summary.csv
- outputs/tables/join_cardinality_audit.csv
- outputs/tables/order_status_summary.csv
- outputs/tables/master_table_validation.csv
- data/processed/master_order_table.csv
- data/processed/analysis_orders.csv
- data/processed/order_seller_bridge.csv
- data/processed/order_category_bridge.csv


## Notebook Summary

本 notebook 完成了项目的 **Level 1 数据基础**：

- 9 张 raw Olist tables 已成功读取并完成 schema / missingness / key audit；
- `orders` 被确认作为 order-level base，最终 master 保持 **99,441 rows = 99,441 unique `order_id`**；
- `order_items`、`order_payments`、`order_reviews` 在进入 master 前分别完成 aggregation / deterministic dedupe，避免 raw one-to-many merge 造成订单重复；
- 数据中存在 multi-seller、multi-category 与 multi-review 情况，因此另外保存 order–seller / order–category bridge，而没有把一个“primary”标签错误当成完整 attribution；
- delivery-analysis population 被单独定义为 delivered 且 actual / estimated delivery date 均可观察的订单，共 **96,470 orders**；
- calendar-date late rule 在这里被统一定义，但 late rate、delay severity 与 review impact 的业务解释留给后续 notebook；
- 所有关键 validation checks 均需 PASS 后才保存 processed files。

### 本 notebook 可以诚实支持的 claim

> Olist raw tables have different grains, including several one-to-many relationships at the order level. Aggregation or deterministic deduplication is therefore required before constructing a one-row-per-order analytical table.

### 本 notebook 不应该支持的 claim

这里还不能说 late delivery **causes** bad reviews、某个 seller 对 multi-seller order 的失败负唯一责任，或 proposed recovery system 已被数据证明有效。

下一步 `02_late_delivery_baseline.ipynb` 应直接读取 `data/processed/analysis_orders.csv`，而不是重新从 raw tables 开始 merge。

## How this notebook builds on the reference projects
The reference projects provided useful methodological patterns for preparing the Olist data, including checking table grain, aggregating one-to-many records before joining, resolving duplicate reviews, and defining delivery delay from order timestamps.

This notebook adopts those principles but applies them within a stricter order-level data contract. It validates join cardinality, preserves one row per order_id, separates the full master table from the delivery-analysis population, and retains order–seller and order–category bridge tables for later analysis.

## What this notebook adds
The main contribution is a reusable and validated data foundation for the post-purchase CX redesign workflow.

It explicitly separates:

direct data processing and validation — establishing reliable order-level analytical data;
later customer-impact analysis — testing how delivery performance relates to reviews and other observed outcomes;
future service-design intervention — translating validated findings into journey-map, service-blueprint, and recovery-design hypotheses that still require future validation.
This separation prevents descriptive data preparation from being presented as causal evidence or intervention validation. It also creates a traceable link between the raw Olist data and the downstream CX redesign work.